### 1) Download Dataset Movielens 20M and Import Lib

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("grouplens/movielens-20m-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/movielens-20m-dataset


In [2]:
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import math

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from scipy.sparse import coo_matrix, csr_matrix


import os


### 2) Read Dataset and data preprocessing

#### 2.1) Tiền xử lý dữ liệu movies

In [3]:
df_movies = pd.read_csv('/kaggle/input/movielens-20m-dataset/movie.csv')
print(df_movies.shape)
df_movies.head()

(27278, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27278 entries, 0 to 27277
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  27278 non-null  int64 
 1   title    27278 non-null  object
 2   genres   27278 non-null  object
dtypes: int64(1), object(2)
memory usage: 639.5+ KB


In [5]:
# Tách năm phát hành ra khỏi cột 'title'
df_movies['year'] = df_movies['title'].str.extract(r'(\d{4})', expand=False)

# Loại bỏ phần năm trong tiêu đề phim
df_movies['title'] = (
    df_movies['title']
    .str.replace(r'\(\d{4}\)', '', regex=True)
    .str.strip()
)
df_movies.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


In [6]:
# Chuyển cột 'genres' thành danh sách các thể loại
# df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|') if isinstance(x, str) else [])

# Ép kiểu dữ liệu cho cột 'year' về numeric
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce')
df_movies.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,2,Jumanji,Adventure|Children|Fantasy,1995.0
2,3,Grumpier Old Men,Comedy|Romance,1995.0
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995.0
4,5,Father of the Bride Part II,Comedy,1995.0


In [7]:
print(df_movies.isnull().sum())

movieId     0
title       0
genres      0
year       19
dtype: int64


In [8]:
df_movies = df_movies.dropna(subset=['year'])
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27259 entries, 0 to 27277
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  27259 non-null  int64  
 1   title    27259 non-null  object 
 2   genres   27259 non-null  object 
 3   year     27259 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 1.0+ MB


#### 2.2) Tiền xử lý dữ liệu Rating

In [9]:
df_ratings = pd.read_csv(
    '/kaggle/input/movielens-20m-dataset/rating.csv',
    usecols=['userId', 'movieId', 'rating', 'timestamp'],
    dtype={'userId': np.int32, 'movieId': np.int32, 'rating': np.float32}
)
print(df_ratings.shape)
df_ratings.head()

(20000263, 4)


,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [10]:
df_ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000263 entries, 0 to 20000262
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int32  
 1   movieId    int32  
 2   rating     float32
 3   timestamp  object 
dtypes: float32(1), int32(2), object(1)
memory usage: 381.5+ MB


In [11]:
print(df_ratings.isnull().sum())

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


#### 2.3) Tiền xử lý dữ liệu df_tags 

In [12]:
# Cột tag trong MovieLens chính là nguồn thông tin ngữ nghĩa (semantic) giúp ta hiểu thêm về ý nghĩa phim ngoài các thể loại (genres)
df_tags = pd.read_csv('/kaggle/input/movielens-20m-dataset/tag.csv')
print(df_tags.shape)
df_tags.head()

(465564, 4)


,userId,movieId,tag,timestamp
0,18,4141,Mark Waters,2009-04-24 18:19:40
1,65,208,dark hero,2013-05-10 01:41:18
2,65,353,dark hero,2013-05-10 01:41:19
3,65,521,noir thriller,2013-05-10 01:39:43
4,65,592,dark hero,2013-05-10 01:41:18


In [13]:
df_tags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 465564 entries, 0 to 465563
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   userId     465564 non-null  int64 
 1   movieId    465564 non-null  int64 
 2   tag        465548 non-null  object
 3   timestamp  465564 non-null  object
dtypes: int64(2), object(2)
memory usage: 14.2+ MB


In [14]:
print(df_tags.isnull().sum())

userId        0
movieId       0
tag          16
timestamp     0
dtype: int64


In [15]:
# Xóa giá trị thiếu hoặc trống trong cột 'tag'
df_tags = df_tags.dropna(subset=['tag'])
df_tags = df_tags[df_tags['tag'].str.strip() != '']

# Chuẩn hóa chữ thường và loại bỏ khoảng trắng thừa trong cột 'tag'
df_tags['tag'] = df_tags['tag'].str.lower().str.strip()
df_tags.info()

<class 'pandas.core.frame.DataFrame'>
Index: 465541 entries, 0 to 465563
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   userId     465541 non-null  int64 
 1   movieId    465541 non-null  int64 
 2   tag        465541 non-null  object
 3   timestamp  465541 non-null  object
dtypes: int64(2), object(2)
memory usage: 17.8+ MB


#### 2.4) Tiền xử lý dữ liệu df_link

In [16]:
# File này là cầu nối (mapping) giữa tập MovieLens và các cơ sở dữ liệu phim lớn khác — đặc biệt là IMDb và TMDb.
# trong đồ án này không dùng đến nên không tiền xử lý gì cả
df_links = pd.read_csv('/kaggle/input/movielens-20m-dataset/link.csv')
print(df_links.shape)
df_links.head()

(27278, 3)


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [17]:
df_links.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27278 entries, 0 to 27277
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  27278 non-null  int64  
 1   imdbId   27278 non-null  int64  
 2   tmdbId   27026 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 639.5 KB


#### 2.5) Tiền xử lý dữ liệu df_genome_scores

In [18]:
# là nền tảng của “Genome Tag Recommender System”, giúp hệ thống hiểu sâu hơn về ý nghĩa nội dung (semantic meaning) của từng phim
# relevance: mức độ liên quan của một thẻ (tag) cụ thể đối với một bộ phim nhất định, với giá trị từ 0 đến 1.

df_genome_scores = pd.read_csv('/kaggle/input/movielens-20m-dataset/genome_scores.csv')
print(df_genome_scores.shape)
df_genome_scores.head()

(11709768, 3)


,movieId,tagId,relevance
0,1,1,0.02500
1,1,2,0.02500
2,1,3,0.05775
3,1,4,0.09675
4,1,5,0.14675


In [19]:
df_genome_scores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11709768 entries, 0 to 11709767
Data columns (total 3 columns):
 #   Column     Dtype  
---  ------     -----  
 0   movieId    int64  
 1   tagId      int64  
 2   relevance  float64
dtypes: float64(1), int64(2)
memory usage: 268.0 MB


In [20]:
# Ép kiểu dữ liệu để giảm bộ nhớ
df_genome_scores = df_genome_scores.astype({
    'movieId': 'int32',
    'tagId': 'int32',
    'relevance': 'float32'
})
df_genome_scores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11709768 entries, 0 to 11709767
Data columns (total 3 columns):
 #   Column     Dtype  
---  ------     -----  
 0   movieId    int32  
 1   tagId      int32  
 2   relevance  float32
dtypes: float32(1), int32(2)
memory usage: 134.0 MB


#### 2.6) Tiền xử lý dữ liệu genome_tags

In [21]:
# genome_tags là từ điển (vocabulary) của toàn bộ 1.128 tag ngữ nghĩa (semantic tags) mà nhóm nghiên cứu MovieLens đã xây dựng.
df_genome_tags = pd.read_csv('/kaggle/input/movielens-20m-dataset/genome_tags.csv')
print(df_genome_tags.shape)
df_genome_tags.head()

(1128, 2)


,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s


In [22]:
df_genome_tags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1128 entries, 0 to 1127
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   tagId   1128 non-null   int64 
 1   tag     1128 non-null   object
dtypes: int64(1), object(1)
memory usage: 17.8+ KB


In [23]:
#  Chuẩn hóa chữ thường và loại khoảng trắng thừa
df_genome_tags['tag'] = df_genome_tags['tag'].str.lower().str.strip()

# Loại bỏ trùng lặp (nếu có)
df_genome_tags = df_genome_tags.drop_duplicates(subset=['tagId'])

df_genome_tags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1128 entries, 0 to 1127
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   tagId   1128 non-null   int64 
 1   tag     1128 non-null   object
dtypes: int64(1), object(1)
memory usage: 17.8+ KB


### 3) Trực quan hóa dữ liệu

### 4) Popular Recommender System Algorithms

Precision@K, Recall@K, NDCG@K… là chỉ số đánh giá kết quả gợi ý, không phụ thuộc loại mô hình
--> 
Dù thuật toán đơn giản hay phức tạp, miễn là đầu ra là danh sách K phim được gợi ý cho mỗi user, đều đánh giá được.

In [24]:
def temporal_train_test_split(df, test_ratio=0.2):
    train_list, test_list = [], []

    for _, user_df in df.groupby('userId'):
        user_df = user_df.sort_values('timestamp')
        n_test = max(1, int(len(user_df) * test_ratio))

        test_list.append(user_df.tail(n_test))
        train_list.append(user_df.iloc[:-n_test])

    train = pd.concat(train_list).reset_index(drop=True)
    test = pd.concat(test_list).reset_index(drop=True)
    return train, test


In [25]:
train, test = temporal_train_test_split(df_ratings, test_ratio=0.2)
train[:5]

,userId,movieId,rating,timestamp
0,1,924,3.5,2004-09-10 03:06:38
1,1,919,3.5,2004-09-10 03:07:01
2,1,2683,3.5,2004-09-10 03:07:30
3,1,1584,3.5,2004-09-10 03:07:36
4,1,1079,4.0,2004-09-10 03:07:45


In [26]:
print(train.shape, test.shape)
test[:5]

(16052927, 4) (3947336, 4)


,userId,movieId,rating,timestamp
0,1,7164,3.5,2005-04-02 23:52:03
1,1,2021,4.0,2005-04-02 23:52:09
2,1,7046,4.0,2005-04-02 23:52:14
3,1,2143,4.0,2005-04-02 23:52:31
4,1,2100,4.0,2005-04-02 23:52:35


In [27]:
def precision_at_k(recs, relevant, k):
    recs = recs[:k]
    return len(set(recs) & set(relevant)) / k

def recall_at_k(recs, relevant, k):
    if len(relevant) == 0:
        return 0
    recs = recs[:k]
    return len(set(recs) & set(relevant)) / len(relevant)

def ndcg_at_k(recs, relevant, k):
    dcg = 0.0
    for i, r in enumerate(recs[:k]):
        if r in relevant:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0

In [28]:
movie_popularity = (train.groupby('movieId').agg(count=('rating', 'count'), avg_rating=('rating', 'mean')))
movie_popularity.sort_values('count', ascending=False)[:5]

,count,avg_rating
movieId,,
296,61324,4.176310
356,59537,4.032115
318,57615,4.452911
593,57339,4.183331
480,53660,3.663278


In [29]:
# Tính điểm score phổ biến
movie_popularity['score'] = (0.7 * movie_popularity['count'] + 0.3 * movie_popularity['avg_rating'])
movie_popularity.sort_values('score', ascending=False)[:5]

,count,avg_rating,score
movieId,,,
296,61324,4.176310,42928.052893
356,59537,4.032115,41677.109634
318,57615,4.452911,40331.835873
593,57339,4.183331,40138.554999
480,53660,3.663278,37563.098984


In [30]:
# Lấy id movie
popular_ranking = (movie_popularity.sort_values('score', ascending=False).index.tolist())
popular_ranking[:5]

[296, 356, 318, 593, 480]

In [31]:
# Hàm gợi ý phương pháp popular
def recommend_popular(user_id, k=10):
    watched = set(train[train['userId'] == user_id]['movieId'])
    recs = [m for m in popular_ranking if m not in watched]
    return recs[:k]


In [32]:
users = test['userId'].unique()

In [33]:
len(users) #138k người dùng

138493

In [34]:
np.random.seed(42)

sample_users = np.random.choice(
    users,
    size=100000,
    replace=False
)
sample_users[:5]

array([ 9761, 28327, 59407, 76329, 10882], dtype=int32)

In [35]:
K = 10
prec_p, rec_p, ndcg_p = [], [], []

for u in sample_users:
    relevant = test[test['userId'] == u]['movieId'].tolist()
    if len(relevant) == 0:
        continue

    recs_pop = recommend_popular(u, K)
    if not recs_pop:
        continue

    prec_p.append(precision_at_k(recs_pop, relevant, K))
    rec_p.append(recall_at_k(recs_pop, relevant, K))
    ndcg_p.append(ndcg_at_k(recs_pop, relevant, K))

print("POPULAR")
print(f"Precision@{K}:{np.mean(prec_p):.4f}")
print(f"Recall@{K}:{np.mean(rec_p):.4f}")
print(f"NDCG@{K}:{np.mean(ndcg_p):.4f}")

POPULAR
Precision@10:0.0812
Recall@10:0.0483
NDCG@10:0.0940


In [36]:
user_id = sample_users[0]
print(f"UserId = {user_id}")

UserId = 9761


In [37]:
watched = train[train['userId'] == user_id].merge(
    df_movies[['movieId', 'title']],
    on='movieId',
    how='left'
)[['movieId', 'title', 'rating']]

print("\nPhim user đã xem (train):")
display(watched.sort_values('rating', ascending=False))


Phim user đã xem (train):


,movieId,title,rating
1,296,Pulp Fiction,5.0
35,50,"Usual Suspects, The",5.0
20,110,Braveheart,5.0
56,555,True Romance,5.0
49,246,Hoop Dreams,5.0
...,...,...,...
27,410,Addams Family Values,1.0
14,329,Star Trek: Generations,1.0
38,151,Rob Roy,1.0
36,282,Nell,1.0


In [38]:
K = 10
recs_pop = recommend_popular(user_id, K)

recs_pop_df = (
    df_movies[df_movies['movieId'].isin(recs_pop)]
    [['movieId', 'title']]
)

print("\nTop phim được gợi ý (POPULAR):")
display(recs_pop_df)



Top phim được gợi ý (POPULAR):


,movieId,title
0,1,Toy Story
257,260,Star Wars: Episode IV - A New Hope
352,356,Forrest Gump
453,457,"Fugitive, The"
476,480,Jurassic Park
523,527,Schindler's List
583,589,Terminator 2: Judgment Day
587,593,"Silence of the Lambs, The"
1184,1210,Star Wars: Episode VI - Return of the Jedi
2486,2571,"Matrix, The"


In [39]:
test_movies = test[test['userId'] == user_id].merge(
    df_movies[['movieId', 'title']],
    on='movieId',
    how='left'
)[['movieId', 'title']]

print("\nPhim user xem trong TEST (ground truth):")
display(test_movies)



Phim user xem trong TEST (ground truth):


,movieId,title
0,427,Boxing Helena
1,45,To Die For
2,260,Star Wars: Episode IV - A New Hope
3,267,Major Payne
4,610,Heavy Metal
5,437,Cops and Robbersons
6,233,Exotica
7,215,Before Sunrise
8,177,Lord of Illusions
9,198,Strange Days


### 5) Content-Based Recommender System

Gợi ý theo nội dung phim (thể loại genres) bằng TF-IDF + cosine similarity,

In [40]:
df_movies_train = (
    df_movies[df_movies['movieId'].isin(train['movieId'].unique())]
    .copy()
    .reset_index(drop=True)
)
df_movies_train['genres_str'] = df_movies_train['genres'].apply(lambda s: s.replace("|", " ").replace("-", ""))
df_movies_train['genres_str'] = df_movies_train['genres_str'].str.replace('(no genres listed)', 'no_genres_listed', regex=False)

In [41]:
df_movies_train.head()

,movieId,title,genres,year,genres_str
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995.0,Adventure Animation Children Comedy Fantasy
1,2,Jumanji,Adventure|Children|Fantasy,1995.0,Adventure Children Fantasy
2,3,Grumpier Old Men,Comedy|Romance,1995.0,Comedy Romance
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995.0,Comedy Drama Romance
4,5,Father of the Bride Part II,Comedy,1995.0,Comedy


In [42]:
# TF-IDF
# tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=2, token_pattern=r"(?u)\b[\w\-]+\b")
# tfidf_matrix = tfidf.fit_transform(df_movies_train['genres_str'])
# tfidf_matrix

In [43]:
tfidf = TfidfVectorizer(ngram_range=(1,1))
tfidf_matrix = tfidf.fit_transform(df_movies_train['genres_str'])
print(tfidf.vocabulary_)
print(len(tfidf.vocabulary_))
print(tfidf_matrix.shape)

{'adventure': 1, 'animation': 2, 'children': 3, 'comedy': 4, 'fantasy': 8, 'romance': 15, 'drama': 7, 'action': 0, 'crime': 5, 'thriller': 17, 'horror': 10, 'mystery': 13, 'scifi': 16, 'imax': 11, 'documentary': 6, 'war': 18, 'musical': 12, 'western': 19, 'filmnoir': 9, 'no_genres_listed': 14}
20
(22098, 20)


In [44]:
# Cosine
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(df_movies_train.index, index=df_movies_train['movieId'])
indices

movieId
1             0
2             1
3             2
4             3
5             4
          ...  
131241    22093
131243    22094
131248    22095
131250    22096
131252    22097
Length: 22098, dtype: int64

In [45]:
cosine_sim

array([[1.        , 0.81360796, 0.16017781, ..., 1.        , 0.2656891 ,
        0.14478785],
       [0.81360796, 1.        , 0.        , ..., 0.81360796, 0.        ,
        0.        ],
       [0.16017781, 0.        , 1.        , ..., 0.16017781, 0.60287686,
        0.32853904],
       ...,
       [1.        , 0.81360796, 0.16017781, ..., 1.        , 0.2656891 ,
        0.14478785],
       [0.2656891 , 0.        , 0.60287686, ..., 0.2656891 , 1.        ,
        0.54495215],
       [0.14478785, 0.        , 0.32853904, ..., 0.14478785, 0.54495215,
        1.        ]])

In [46]:
def recommend_content(user_id, k=10, top_n_seed=10):
    user_hist = train[train['userId'] == user_id]
    if len(user_hist) == 0:
        return []

    top_movies = (user_hist.sort_values('rating', ascending=False).head(top_n_seed))

    seed_indices = [indices[m] for m in top_movies['movieId'] if m in indices]
    if len(seed_indices) == 0:
        print("bị 0")
        return []

    weights = top_movies['rating'].values[:len(seed_indices)]

    sim_scores = np.average(cosine_sim[seed_indices], axis=0, weights=weights)
    # sim_scores = cosine_sim[seed_indices].max(axis=0)


    ranked_idx = np.argsort(sim_scores)[::-1]

    watched = set(user_hist['movieId'])
    recs = []

    for idx in ranked_idx:
        mid = df_movies_train.iloc[idx]['movieId']
        if mid not in watched:
            recs.append(mid)
        if len(recs) == k:
            break

    return recs


In [47]:
np.random.seed(42)

sample_users = np.random.choice(
    users,
    size=100000,
    replace=False
)

In [48]:
K = 10
prec_c, rec_c, ndcg_c = [], [], []

for u in sample_users:
    relevant = test[test['userId'] == u]['movieId'].tolist()
    if len(relevant) == 0:
        continue

    recs_cont = recommend_content(u, K)
    if not recs_cont:
        continue

    prec_c.append(precision_at_k(recs_cont, relevant, K))
    rec_c.append(recall_at_k(recs_cont, relevant, K))
    ndcg_c.append(ndcg_at_k(recs_cont, relevant, K))

print("CONTENT BASED WITH 100000 USER")
print(f"Precision@{K}:   {np.mean(prec_c):.4f}")
print(f"Recall@{K}:      {np.mean(rec_c):.4f}")
print(f"NDCG@{K}:        {np.mean(ndcg_c):.4f}")

CONTENT BASED WITH 100000 USER
Precision@10:   0.0056
Recall@10:      0.0026
NDCG@10:        0.0060


In [49]:
# số interaction trong train của mỗi user test
user_train_counts = train.groupby('userId').size()

test_users = test['userId'].unique()

cold_stats = pd.Series(test_users).map(user_train_counts).fillna(0)

cold_stats.describe()

count    138493.000000
mean        115.911468
std         184.221930
min          16.000000
25%          28.000000
50%          55.000000
75%         124.000000
max        7404.000000
dtype: float64

In [50]:
user_id = sample_users[0]   
print(f"UserId = {user_id}")

watched_df = (
    train[train['userId'] == user_id]
    .merge(df_movies[['movieId', 'title', 'genres']], on='movieId', how='left')
    .sort_values('rating', ascending=False)
)

print("\nPhim người dùng đã xem (TRAIN):")
display(watched_df[['movieId', 'title', 'genres', 'rating']].head(30))

UserId = 9761

Phim người dùng đã xem (TRAIN):


,movieId,title,genres,rating
1,296,Pulp Fiction,Comedy|Crime|Drama|Thriller,5.0
35,50,"Usual Suspects, The",Crime|Mystery|Thriller,5.0
20,110,Braveheart,Action|Drama|War,5.0
56,555,True Romance,Crime|Thriller,5.0
49,246,Hoop Dreams,Documentary,5.0
9,318,"Shawshank Redemption, The",Crime|Drama,5.0
22,32,Twelve Monkeys (a.k.a. 12 Monkeys),Mystery|Sci-Fi|Thriller,4.0
5,588,Aladdin,Adventure|Animation|Children|Comedy|Musical,4.0
47,204,Under Siege 2: Dark Territory,Action,4.0
63,608,Fargo,Comedy|Crime|Drama|Thriller,4.0


In [51]:
recs_con = recommend_content(user_id, K)

recs_con_df = (
    df_movies[df_movies['movieId'].isin(recs_con)]
    [['movieId', 'title', 'genres']]
)

print("\nGợi ý theo CONTENT-BASED:")
display(recs_con_df)



Gợi ý theo CONTENT-BASED:


,movieId,title,genres
19,20,Money Train,Action|Comedy|Crime|Drama|Thriller
143,145,Bad Boys,Action|Comedy|Crime|Drama|Thriller
1398,1432,Metro,Action|Comedy|Crime|Drama|Thriller
3188,3275,"Boondock Saints, The",Action|Crime|Drama|Thriller
4931,5027,Another 48 Hrs.,Action|Comedy|Crime|Drama|Thriller
5529,5628,Wasabi,Action|Comedy|Crime|Drama|Thriller
6895,7007,"Last Boy Scout, The",Action|Comedy|Crime|Drama|Thriller
10842,43853,"Business, The",Action|Comedy|Crime|Drama|Thriller
12974,61553,"Fifth Commandment, The",Action|Comedy|Crime|Drama|Thriller
20790,101715,Loaded,Action|Crime|Drama|Thriller


In [52]:
test_df = (
    test[test['userId'] == user_id]
    .merge(df_movies[['movieId', 'title', 'genres']], on='movieId', how='left')
)

print("\nPhim người dùng xem trong TEST (ground truth):")
display(test_df[['movieId', 'title', 'genres']])



Phim người dùng xem trong TEST (ground truth):


,movieId,title,genres
0,427,Boxing Helena,Drama|Mystery|Romance|Thriller
1,45,To Die For,Comedy|Drama|Thriller
2,260,Star Wars: Episode IV - A New Hope,Action|Adventure|Sci-Fi
3,267,Major Payne,Comedy
4,610,Heavy Metal,Action|Adventure|Animation|Horror|Sci-Fi
5,437,Cops and Robbersons,Comedy
6,233,Exotica,Drama
7,215,Before Sunrise,Drama|Romance
8,177,Lord of Illusions,Horror
9,198,Strange Days,Action|Crime|Drama|Mystery|Sci-Fi|Thriller


### 6) Matrix factorization (MF)

In [53]:
from scipy.sparse import coo_matrix
from sklearn.decomposition import TruncatedSVD

In [54]:
user_ids = train['userId'].unique()
movie_ids = train['movieId'].unique()

user2idx = {u: i for i, u in enumerate(user_ids)}
idx2user = {i: u for u, i in user2idx.items()}

movie2idx = {m: i for i, m in enumerate(movie_ids)}
idx2movie = {i: m for m, i in movie2idx.items()}


In [55]:
rows = train['userId'].map(user2idx)
cols = train['movieId'].map(movie2idx)
data = train['rating'].values

R = coo_matrix(
    (data, (rows, cols)),
    shape=(len(user2idx), len(movie2idx))
)

R

<COOrdinate sparse matrix of dtype 'float32'
	with 16052927 stored elements and shape (138493, 22107)>

In [56]:
n_factors = 100

svd = TruncatedSVD(
    n_components=n_factors,
    random_state=42
)

U = svd.fit_transform(R)        # user latent factors
V = svd.components_             # item latent factors

In [57]:
U

array([[ 1.9071732e+01, -5.8186417e+00,  1.2814615e+00, ...,
         1.4986016e+00,  3.8939385e+00,  1.9821756e+00],
       [ 4.6996226e+00,  1.3113364e+00, -7.2573781e-01, ...,
         9.5879680e-01,  3.3905804e-01,  1.8035253e+00],
       [ 2.3076139e+01, -2.6610446e+00, -1.1062684e+01, ...,
         2.3599696e-01,  2.3718536e+00, -3.1096977e-01],
       ...,
       [ 6.8383259e-01, -8.3892025e-02, -5.0040603e-01, ...,
        -7.5830854e-03, -1.8117586e-01, -2.3928404e-01],
       [ 9.7541180e+00, -5.0253658e+00, -8.7377107e-01, ...,
         3.4580252e+00, -3.3884873e+00, -1.9373704e-01],
       [ 2.7499063e+01, -4.6449170e+00,  9.3563967e+00, ...,
         9.7591603e-01, -2.7349496e-01, -1.8499433e+00]], dtype=float32)

In [58]:
def recommend_mf(user_id, k=10):
    if user_id not in user2idx:
        return []

    u_idx = user2idx[user_id]
    user_vector = U[u_idx]

    scores = user_vector @ V   # .product

    watched = set(train[train['userId'] == user_id]['movieId'])

    ranked_idx = np.argsort(scores)[::-1]

    recs = []
    for idx in ranked_idx:
        mid = idx2movie[idx]
        if mid not in watched:
            recs.append(mid)
        if len(recs) == k:
            break

    return recs


In [59]:
K = 10
prec_mf, rec_mf, ndcg_mf = [], [], []

for u in sample_users:
    if u not in user2idx:
        continue

    relevant = test[test['userId'] == u]['movieId'].tolist()
    if not relevant:
        continue

    recs = recommend_mf(u, K)
    if not recs:
        continue

    prec_mf.append(precision_at_k(recs, relevant, K))
    rec_mf.append(recall_at_k(recs, relevant, K))
    ndcg_mf.append(ndcg_at_k(recs, relevant, K))

print("\nMATRIX FACTORIZATION (MF) with 100000usser")
print(f"Precision@{K}:   {np.mean(prec_mf):.4f}")
print(f"Recall@{K}:      {np.mean(rec_mf):.4f}")
print(f"NDCG@{K}:        {np.mean(ndcg_mf):.4f}")


MATRIX FACTORIZATION (MF) with 100000usser
Precision@10:   0.1237
Recall@10:      0.0815
NDCG@10:        0.1422


In [60]:
user_id = sample_users[2]
train_df = (
    train[train['userId'] == user_id]
    .merge(df_movies[['movieId', 'title', 'genres']], on='movieId', how='left')
    .sort_values('rating', ascending=False)
)

print("\nPhim người dùng đã xem trong TRAIN:")
print(f"UserId = {user_id}")
display(train_df[['movieId', 'title', 'genres', 'rating']].head(10))


Phim người dùng đã xem trong TRAIN:
UserId = 59407


,movieId,title,genres,rating
186,7076,Bullitt,Action|Crime|Drama|Thriller,5.0
313,1997,"Exorcist, The",Horror|Mystery,5.0
19,1262,"Great Escape, The",Action|Adventure|Drama|War,5.0
46,1178,Paths of Glory,Drama|War,5.0
611,4916,Midway,Drama|War,5.0
386,919,"Wizard of Oz, The",Adventure|Children|Fantasy|Musical,5.0
453,954,Mr. Smith Goes to Washington,Drama,5.0
183,7980,"Bridge Too Far, A",Action|Drama|War,5.0
949,3524,Arthur,Comedy|Romance,5.0
431,5385,"Last Waltz, The",Documentary,5.0


In [61]:
K = 10
recs_mf = recommend_mf(user_id, K)

mf_recs_df = (
    df_movies[df_movies['movieId'].isin(recs_mf)]
    [['movieId', 'title', 'genres']]
)

print("\nGợi ý theo MATRIX FACTORIZATION (MF):")
display(mf_recs_df)



Gợi ý theo MATRIX FACTORIZATION (MF):


,movieId,title,genres
149,151,Rob Roy,Action|Drama|Romance|War
897,914,My Fair Lady,Comedy|Drama|Musical|Romance
923,940,"Adventures of Robin Hood, The",Action|Adventure|Romance
1064,1086,Dial M for Murder,Crime|Mystery|Thriller
2281,2366,King Kong,Action|Adventure|Fantasy|Horror
2831,2917,Body Heat,Crime|Thriller
3362,3451,Guess Who's Coming to Dinner,Drama
4129,4223,Enemy at the Gates,Drama|War
6697,6807,Monty Python's The Meaning of Life,Comedy
7332,7458,Troy,Action|Adventure|Drama|War


In [62]:
test_df = (
    test[test['userId'] == user_id]
    .merge(df_movies[['movieId', 'title', 'genres']], on='movieId', how='left')
)

print("\nPhim người dùng xem trong TEST (ground truth):")
display(test_df[['movieId', 'title', 'genres']])



Phim người dùng xem trong TEST (ground truth):


,movieId,title,genres
0,1299,"Killing Fields, The",Drama|War
1,2726,"Killing, The",Crime|Film-Noir
2,3137,"Sea Wolves, The",Action|War
3,1942,All the King's Men,Drama
4,7215,To Have and Have Not,Adventure|Drama|Romance|Thriller|War
...,...,...,...
480,82095,Skyline,Sci-Fi|Thriller
481,94896,Bernie,Comedy|Crime|Drama
482,27689,Crimson Rivers 2: Angels of the Apocalypse (Ri...,Action|Crime|Thriller
483,101076,G.I. Joe: Retaliation,Action|Adventure|Sci-Fi|Thriller|IMAX


### 7) Taxonomy-aware

In [63]:
from collections import Counter
from collections import defaultdict

In [64]:
df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|') if isinstance(x, str) else [])

In [65]:
df_movies.head()

,movieId,title,genres,year
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995.0
2,3,Grumpier Old Men,"[Comedy, Romance]",1995.0
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995.0
4,5,Father of the Bride Part II,[Comedy],1995.0


In [66]:
# Lấy all genre

all_genres = set()
for genres in df_movies['genres']:
    all_genres.update(genres)

sorted(all_genres)
# {'adventure': 1, 'animation': 2, 'children': 3, 'comedy': 4, 'fantasy': 8, 'romance': 15, 'drama': 7, 
#  'action': 0, 'crime': 5, 'thriller': 17, 'horror': 10, 'mystery': 13, 'scifi': 16, 'imax': 11, 'documentary': 6,
#  'war': 18, 'musical': 12, 'western': 19, 'filmnoir': 9, 'no_genres_listed': 14}
# 20


['(no genres listed)',
 'Action',
 'Adventure',
 'Animation',
 'Children',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Fantasy',
 'Film-Noir',
 'Horror',
 'IMAX',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Thriller',
 'War',
 'Western']

In [67]:
# Cấp 1
parent_genres = [
    "Action",
    "Comedy",
    "Drama",
    "Fantasy",
    "Horror",
    "Animation",
    "Documentary",
    "Western",
    "Other"
]

In [68]:
# Cấp 2
genre_to_parent = {
    "(no genres listed)": "Other",

    "Action": "Action",
    "Adventure": "Action",
    "Sci-Fi": "Action",
    "War": "Action",
    "IMAX": "Action",

    "Comedy": "Comedy",
    "Romance": "Comedy",
    "Musical": "Comedy",
    "Children": "Comedy",

    "Drama": "Drama",
    "Crime": "Drama",
    "Thriller": "Drama",
    "Mystery": "Drama",
    "Film-Noir": "Drama",

    "Fantasy": "Fantasy",
    "Horror": "Horror",
    "Animation": "Animation",
    "Documentary": "Documentary",
    "Western": "Western"
}


In [69]:
# mapping phim → parent genres (Level 1)
def get_parent_genres(genres_lvl2):
    return set(
        genre_to_parent[g]
        for g in genres_lvl2
        if g in genre_to_parent
    )

In [70]:
get_parent_genres('Animation')

set()

In [71]:
df_movies.head()

,movieId,title,genres,year
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995.0
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995.0
2,3,Grumpier Old Men,"[Comedy, Romance]",1995.0
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995.0
4,5,Father of the Bride Part II,[Comedy],1995.0


In [72]:
# movieId -> set(genres)
movie_genres = {
    row.movieId: set(row.genres)
    for _, row in df_movies.iterrows()
}

In [73]:
# Xây User Profile ở level 1 (parent)
def build_user_parent_profile(user_id):
    user_hist = train[train['userId'] == user_id]
    counter = Counter()

    for _, row in user_hist.iterrows():
        mid = row['movieId']
        rating = row['rating']
        parents = get_parent_genres(movie_genres.get(mid, []))

        for p in parents:
            counter[p] += rating

    total = sum(counter.values())
    if total == 0:
        return {}

    # normalize -> distribution
    return {g: c / total for g, c in counter.items()}

In [74]:
build_user_parent_profile(user_id)

{'Animation': 0.031739686600575634,
 'Comedy': 0.22505596418292292,
 'Action': 0.2689478733610489,
 'Drama': 0.33766389510713146,
 'Fantasy': 0.05996162456028142,
 'Western': 0.03945474896066518,
 'Horror': 0.03249920051167253,
 'Documentary': 0.004677006715701951}

In [75]:
user_tax_cache = {} #cache user profile (tăng tốc)

def get_user_parent_profile(user_id):
    if user_id not in user_tax_cache:
        user_tax_cache[user_id] = build_user_parent_profile(user_id)
    return user_tax_cache[user_id]

In [76]:
# PRECOMPUTE: PARENT → MOVIES
parent_to_movies = defaultdict(set)

for mid, genres in movie_genres.items():
    parents = get_parent_genres(genres)
    for p in parents:
        parent_to_movies[p].add(mid)

In [77]:
# Tính score level1 -item
def taxonomy_score(user_pref, movie_id):
    parents = get_parent_genres(movie_genres.get(movie_id, []))
    return sum(user_pref.get(p, 0) for p in parents)


In [78]:
# Tính content score (level 2)
def content_score(user_hist, movie_id, top_n=5):
    if movie_id not in indices:
        return 0

    top_movies = (
        user_hist.sort_values('rating', ascending=False)
        .head(top_n)['movieId']
    )

    sims = []
    for mid in top_movies:
        if mid in indices:
            sims.append(
                cosine_sim[indices[mid], indices[movie_id]]
            )

    return np.mean(sims) if sims else 0


In [79]:
def retrieve_candidates(user_pref, max_cand=1000):
    # lấy top-2 parent genres
    top_parents = sorted(
        user_pref.items(),
        key=lambda x: x[1],
        reverse=True
    )[:2]

    candidates = set()
    for p, _ in top_parents:
        candidates |= parent_to_movies[p]

    return list(candidates)[:max_cand]


In [80]:
# def recommend_taxonomy_2level(user_id, k=10, alpha=0.7, beta=0.3):
#     user_hist = train[train['userId'] == user_id]
#     if user_hist.empty:
#         return []

#     user_pref = build_user_parent_profile(user_id)
#     watched = set(user_hist['movieId'])

#     scored = []

#     for mid in indices.keys():
#         if mid in watched:
#             continue

#         c_score = content_score(user_hist, mid)
#         t_score = taxonomy_score(user_pref, mid)

#         final_score = alpha * c_score + beta * t_score
#         scored.append((mid, final_score))

#     scored.sort(key=lambda x: x[1], reverse=True)
#     return [m for m, _ in scored[:k]]



def recommend_taxonomy_2level(
    user_id,
    k=10,
    alpha=0.7,
    beta=0.3,
    max_cand=1005
):
    user_hist = train[train['userId'] == user_id]
    if user_hist.empty:
        return []

    user_pref = get_user_parent_profile(user_id)
    watched = set(user_hist['movieId'])

    # ===== Stage 1: Retrieve =====
    candidates = retrieve_candidates(user_pref, max_cand)

    scored = []

    # ===== Stage 2: Rerank =====
    for mid in candidates:
        if mid in watched or mid not in indices:
            continue

        c_score = content_score(user_hist, mid)
        t_score = taxonomy_score(user_pref, mid)

        final_score = alpha * c_score + beta * t_score
        scored.append((mid, final_score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return [m for m, _ in scored[:k]]


In [81]:
np.random.seed(42)

sample_users = np.random.choice(
    users,
    size=1008,
    replace=False
)

In [82]:
gprec, grec, gndcg = [], [], []
K = 10

for u in sample_users:
    relevant = test[test['userId'] == u]['movieId'].tolist()
    if not relevant:
        continue

    recs = recommend_taxonomy_2level(u, K)
    if not recs:
        continue

    gprec.append(precision_at_k(recs, relevant, K))
    grec.append(recall_at_k(recs, relevant, K))
    gndcg.append(ndcg_at_k(recs, relevant, K))

print("TAXONOMY-AWARE 2-LEVEL")
print(f"Precision@{K}:  {np.mean(gprec):.4f}")
print(f"Recall@{K}:     {np.mean(grec):.4f}")
print(f"NDCG@{K}:       {np.mean(gndcg):.4f}")


TAXONOMY-AWARE 2-LEVEL
Precision@10:  0.0153
Recall@10:     0.0090
NDCG@10:       0.0163


In [83]:
user_id = sample_users[0]
train_df = (
    train[train['userId'] == user_id]
    .merge(df_movies[['movieId', 'title', 'genres']], on='movieId', how='left')
    .sort_values('rating', ascending=False)
)

print("\nPhim người dùng đã xem trong TRAIN:")
print(f"UserId = {user_id}")
display(train_df[['movieId', 'title', 'genres', 'rating']].head(10))



Phim người dùng đã xem trong TRAIN:
UserId = 9761


,movieId,title,genres,rating
1,296,Pulp Fiction,"[Comedy, Crime, Drama, Thriller]",5.0
35,50,"Usual Suspects, The","[Crime, Mystery, Thriller]",5.0
20,110,Braveheart,"[Action, Drama, War]",5.0
56,555,True Romance,"[Crime, Thriller]",5.0
49,246,Hoop Dreams,[Documentary],5.0
9,318,"Shawshank Redemption, The","[Crime, Drama]",5.0
22,32,Twelve Monkeys (a.k.a. 12 Monkeys),"[Mystery, Sci-Fi, Thriller]",4.0
5,588,Aladdin,"[Adventure, Animation, Children, Comedy, Musical]",4.0
47,204,Under Siege 2: Dark Territory,[Action],4.0
63,608,Fargo,"[Comedy, Crime, Drama, Thriller]",4.0


In [84]:
recs_tax = recommend_taxonomy_2level(user_id, 10)

print("\nPhim gợi ý bằng taxonomy:")
display(
    df_movies[df_movies['movieId'].isin(recs_tax)]
    [['movieId', 'title', 'genres']]
)


Phim gợi ý bằng taxonomy:


,movieId,title,genres
19,20,Money Train,"[Action, Comedy, Crime, Drama, Thriller]"
77,78,"Crossing Guard, The","[Action, Crime, Drama, Thriller]"
143,145,Bad Boys,"[Action, Comedy, Crime, Drama, Thriller]"
416,420,Beverly Hills Cop III,"[Action, Comedy, Crime, Thriller]"
455,459,"Getaway, The","[Action, Adventure, Crime, Drama, Romance, Thr..."
846,861,Supercop (Police Story 3: Supercop) (Jing cha ...,"[Action, Comedy, Crime, Thriller]"
860,876,Supercop 2 (Project S) (Chao ji ji hua),"[Action, Comedy, Crime, Thriller]"
977,996,Last Man Standing,"[Action, Crime, Drama, Thriller]"
13340,65596,Mesrine: Killer Instinct (L'instinct de mort),"[Action, Crime, Drama, Thriller]"
13436,66302,"Bandit, The (Eskiya)","[Action, Crime, Romance, Thriller]"


In [85]:
test_df = (
    test[test['userId'] == user_id]
    .merge(df_movies[['movieId', 'title', 'genres']], on='movieId', how='left')
)

print("\nPhim người dùng xem trong TEST (ground truth):")
display(test_df[['movieId', 'title', 'genres']])


Phim người dùng xem trong TEST (ground truth):


,movieId,title,genres
0,427,Boxing Helena,"[Drama, Mystery, Romance, Thriller]"
1,45,To Die For,"[Comedy, Drama, Thriller]"
2,260,Star Wars: Episode IV - A New Hope,"[Action, Adventure, Sci-Fi]"
3,267,Major Payne,[Comedy]
4,610,Heavy Metal,"[Action, Adventure, Animation, Horror, Sci-Fi]"
5,437,Cops and Robbersons,[Comedy]
6,233,Exotica,[Drama]
7,215,Before Sunrise,"[Drama, Romance]"
8,177,Lord of Illusions,[Horror]
9,198,Strange Days,"[Action, Crime, Drama, Mystery, Sci-Fi, Thriller]"
